In [2]:
import pandas as pd
import re

df=pd.read_csv("train.csv")
mots_inutiles = set([
    "i","me","my","myself","we","our","ours","ourselves",
    "you","your","yours","yourself","yourselves",
    "he","him","his","himself",
    "she","her","hers","herself",
    "it","its","itself",
    "they","them","their","theirs","themselves",
    "what","which","who","whom","this","that","these","those",
    "am","is","are","was","were","be","been","being",
    "have","has","had","do","does","did",
    "a","an","the","and","or","but","if","because","as",
    "until","while","of","at","by","for","with","about","against",
    "between","into","through","during","before","after","above","below",
    "to","from","up","down","in","out","on","off","over","under",
    "again","further","then","once","get", "just", "like", "can", "not"
])
contractions = {
    "don't":"do not",
    "can't":"can not",
    "i'm":"i am",
    "it's":"it is",
    "i've":"i have",
    "you're":"you are",
    "we're":"we are",
    "they're":"they are"
}
def preprocess(texte):
    if pd.isna(texte):
        return []

    texte=texte.lower()
    for k,v in contractions.items():
        texte=texte.replace(k, v)
    texte=re.sub(r'[^\w\s]',' ',texte)

    tokens=texte.split()

    tokens=[t for t in tokens if t not in mots_inutiles]
    tokens=[t for t in tokens if len(t) > 2]

    return tokens

df['patient_clean'] = df.iloc[:,0].apply(preprocess)
df['therapist_clean'] = df.iloc[:,1].apply(preprocess)
nb_patients=df.iloc[:,0].nunique()
print("Patients uniques:", nb_patients)
nb_therapeutes=df.iloc[:,1].nunique()
print("Thérapeutes uniques:", nb_therapeutes)


echange_moyens = len(df) / nb_patients
print("Échanges moyens par patient:", echange_moyens)



Patients uniques: 995
Thérapeutes uniques: 2479
Échanges moyens par patient: 3.52964824120603


In [3]:
from collections import Counter

tout_mots=[mot for tokens in df['patient_clean'] for mot in tokens]
freq = Counter(tout_mots)

print(freq.most_common(20))

[('how', 1272), ('feel', 1223), ('know', 974), ('want', 768), ('when', 762), ('years', 661), ('time', 635), ('never', 623), ('now', 567), ('all', 554), ('always', 509), ('really', 508), ('relationship', 504), ('should', 461), ('think', 442), ('people', 434), ('love', 428), ('still', 424), ('even', 421), ('anxiety', 412)]


In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 48.8 MB/s eta 0:00:00


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
df['patient_text'] = df['patient_clean'].apply(lambda x: " ".join(x))

tfidf = TfidfVectorizer(max_features=2000)

X = tfidf.fit_transform(df['patient_text'])
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster_tfidf'] = kmeans.fit_predict(X)
from gensim.models import Word2Vec

model = Word2Vec(sentences=df['patient_clean'], vector_size=100, window=5, min_count=2)
model.wv.most_similar("feel")

[('wrong', 0.9928716421127319),
 ('say', 0.9924267530441284),
 ('else', 0.9915610551834106),
 ('talk', 0.9900649785995483),
 ('know', 0.9894655346870422),
 ('someone', 0.9892120361328125),
 ('want', 0.9873777627944946),
 ('sometimes', 0.9873174428939819),
 ('why', 0.9871508479118347),
 ('anything', 0.9852144718170166)]

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
vectorizer = CountVectorizer(max_features=2000)
X = vectorizer.fit_transform(df['patient_text'])
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

TypeError: CountVectorizer.fit_transform() got an unexpected keyword argument 'stop_words'

In [7]:
mots=vectorizer.get_feature_names_out()

for i, sujet in enumerate(lda.components_):
    print(f"Sujet {i}")
    print([mots[i] for i in sujet.argsort()[-10:]])

Sujet 0
['depression', 'still', 'address', 'history', 'many', 'know', 'should', 'issues', 'counseling', 'how']
Sujet 1
['think', 'really', 'years', 'help', 'time', 'always', 'when', 'know', 'all', 'feel']
Sujet 2
['having', 'time', 'people', 'something', 'always', 'when', 'want', 'never', 'how', 'feel']
Sujet 3
['years', 'there', 'know', 'very', 'relationship', 'past', 'want', 'now', 'feel', 'child']
Sujet 4
['time', 'now', 'years', 'someone', 'feel', 'sex', 'love', 'want', 'know', 'how']
